In [12]:
import torch
from PIL import Image
import requests
# ViLT 모델과 프로세서를 임포트합니다.
from transformers import ViltProcessor, ViltForQuestionAnswering
import io

# --- 1. 모델과 프로세서 로드 ---
# VQA 데이터셋으로 사전 학습된 ViLT 모델을 로드합니다.
# ViltProcessor는 이미지와 텍스트 전처리를 모두 담당합니다.
model_name = "dandelin/vilt-b32-finetuned-vqa"
processor = ViltProcessor.from_pretrained(model_name)
model = ViltForQuestionAnswering.from_pretrained(model_name)

In [13]:
# --- 2. 추론할 데이터 준비 ---
# 사용자께서 수정한 데이터
try:
    image_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg"
    image = Image.open(image_path).convert("RGB")
    question = "What might be the purpose of the person's workout in the image?"
    choices = ["Building muscle and strength", "Practicing for a marathon", "Training for a cycling race", "Preparing for a swimming competition"]
    print(f"이미지 경로: {image_path}")

except FileNotFoundError:
    print(f"오류: {image_path} 에서 이미지를 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

print(f"질문: {question}")
print(f"선택지: {choices}")


# --- 3. 이미지와 텍스트 전처리 ---
# ViltProcessor는 이미지와 텍스트를 한번에 올바르게 처리해줍니다.
encoding = processor(image, question, return_tensors="pt")

이미지 경로: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg
질문: What might be the purpose of the person's workout in the image?
선택지: ['Building muscle and strength', 'Practicing for a marathon', 'Training for a cycling race', 'Preparing for a swimming competition']


In [14]:
# --- 4. 모델 추론 및 4지선다 답변 선택 ---
# 모델에 전처리된 입력을 전달하여 출력을 받습니다.
outputs = model(**encoding)
logits = outputs.logits

# 모델이 학습한 전체 답변 후보(3129개)에 대한 점수가 logits에 들어있습니다.
# 이 중에서 우리가 제시한 4개 선택지의 점수만 비교해야 합니다.

# 모델의 설정(config)에서 '인덱스 -> 답변' 맵을 가져옵니다.
idx_to_label = model.config.id2label

# 거꾸로 '답변 -> 인덱스' 맵을 만듭니다.
label_to_idx = {label: idx for idx, label in idx_to_label.items()}

# 4개 선택지의 점수를 추출합니다.
choice_logits = []
for choice in choices:
    # 선택지를 소문자로 바꿔서 찾습니다.
    choice_text = choice.lower()
    
    # 선택지가 모델의 답변 후보에 있는지 확인하고, 있다면 해당 인덱스를 가져옵니다.
    if choice_text in label_to_idx:
        choice_idx = label_to_idx[choice_text]
        # 해당 인덱스의 logit(점수)를 저장합니다.
        choice_logits.append(logits[0, choice_idx])
    else:
        # 만약 선택지가 모델의 답변 후보에 없다면, 매우 낮은 점수를 부여합니다.
        choice_logits.append(torch.tensor(float('-inf')))
        print(f"주의: '{choice}'는 모델의 답변 후보에 없습니다.")

주의: 'Building muscle and strength'는 모델의 답변 후보에 없습니다.
주의: 'Practicing for a marathon'는 모델의 답변 후보에 없습니다.
주의: 'Training for a cycling race'는 모델의 답변 후보에 없습니다.
주의: 'Preparing for a swimming competition'는 모델의 답변 후보에 없습니다.


In [15]:
# --- 5. 결과 출력 ---
# 추출한 4개의 점수 중 가장 높은 점수의 인덱스를 찾습니다.
if not choice_logits or all(l.isinf() for l in choice_logits):
    predicted_answer = "답변을 찾을 수 없습니다. 모든 선택지가 모델의 답변 후보에 없습니다."
else:
    # 리스트를 텐서로 변환하여 argmax를 사용합니다.
    choice_logits_tensor = torch.stack(choice_logits)
    best_choice_index = torch.argmax(choice_logits_tensor).item()
    predicted_answer = choices[best_choice_index]

print("-" * 30)
print(f"모델 예측 답변: {predicted_answer}")
print("-" * 30)

------------------------------
모델 예측 답변: 답변을 찾을 수 없습니다. 모든 선택지가 모델의 답변 후보에 없습니다.
------------------------------


In [17]:
import torch
from PIL import Image
# CLIP 모델과 프로세서를 임포트합니다.
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification
import requests
import io

# --- 1. 모델과 프로세서 로드 ---
# 제로샷 이미지 분류에 특화된 CLIP 모델을 로드합니다.
model_name = "openai/clip-vit-base-patch32"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForZeroShotImageClassification.from_pretrained(model_name)


# --- 2. 추론할 데이터 준비 ---
try:
    image_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg"
    image = Image.open(image_path).convert("RGB")
    
    # 질문의 맥락을 보기에 추가해주면 성능이 더 좋아질 수 있습니다.
    question = "What might be the purpose of the person's workout in the image?"
    choices = [
        "Building muscle and strength", 
        "Practicing for a marathon", 
        "Training for a cycling race", 
        "Preparing for a swimming competition"
    ]
    
    # 모델에 입력할 텍스트 레이블 (선택지 그대로 사용)
    candidate_labels = choices

    print(f"이미지 경로: {image_path}")
    print(f"질문: {question}")
    print(f"선택지: {candidate_labels}")

except FileNotFoundError:
    print(f"오류: {image_path} 에서 이미지를 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()


# --- 3. 이미지와 텍스트 레이블 전처리 및 추론 ---
# processor가 이미지와 모든 텍스트 레이블을 한번에 처리합니다.
inputs = processor(images=image, text=candidate_labels, return_tensors="pt", padding=True)

# 모델에 입력을 전달하여 출력을 받습니다.
with torch.no_grad():
    outputs = model(**inputs)

# outputs.logits_per_image는 이미지 1개에 대한 각 텍스트 레이블의 유사도 점수입니다.
# shape: (1, num_labels), 이 경우 (1, 4)가 됩니다.
logits_per_image = outputs.logits_per_image 

# --- 4. 결과 출력 ---
# 4개의 점수 중 가장 높은 값의 인덱스를 찾습니다.
predicted_index = logits_per_image.argmax(-1).item()
predicted_answer = candidate_labels[predicted_index]

print("-" * 30)
print(f"모델 예측 답변: {predicted_answer}")
print("-" * 30)

# (참고) 각 선택지별 점수 확인
print("각 선택지별 유사도 점수 (logits):")
# Softmax를 적용하여 확률값으로 변환
probs = logits_per_image.softmax(dim=1) 
for i, choice in enumerate(choices):
    print(f"- {choice}: {probs[0, i].item():.4f}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


이미지 경로: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg
질문: What might be the purpose of the person's workout in the image?
선택지: ['Building muscle and strength', 'Practicing for a marathon', 'Training for a cycling race', 'Preparing for a swimming competition']
------------------------------
모델 예측 답변: Building muscle and strength
------------------------------
각 선택지별 유사도 점수 (logits):
- Building muscle and strength: 0.8680
- Practicing for a marathon: 0.0197
- Training for a cycling race: 0.1027
- Preparing for a swimming competition: 0.0096
